# Porto Taxi — Popular Long Sub-Routes, Activity Zones & Anomalies

Interactive demo of the Spark pipeline's results, for **Colab Enterprise**.

It reads the small **result CSVs** produced by the mining jobs — top-100 routes
for each method at each length configuration, activity zones, anomalies — **not**
the raw 1.9 GB data, so it renders instantly.

Pipeline that produced these files (see `docs/DATAPROC.md`):
`clean_data → feature_engineering → spatial_encoding →
route_mining_{suffix_array, maximal, clustering, graph} → anomaly_analysis`.

**Methods**

| | method | how it finds routes |
|---|---|---|
| A | clustering | MinHash-LSH over directed bigram shingles → star clustering → the longest cell run its members share |
| B | maximal-frequent | contiguous n-gram support table → maximal among those clearing X%, X calibrated per length |
| C | transition graph | PageRank activity zones + dominant-flow heavy paths, validated against real trips |
| D | suffix array | generalised suffix array + LCP intervals — exact, without enumerating windows |

All four report the same unit: a contiguous sub-route with a distinct-trip support.

In [ ]:
# 1. Dependencies (Colab has pandas; add folium + h3)
!pip -q install folium==0.16.0 h3==3.7.7 gcsfs
import pandas as pd, folium, h3

In [ ]:
# 2. Where the result CSVs live.
#    Colab Enterprise authenticates to GCS automatically, so a gs:// path works
#    directly. Locally, point BASE at ./outputs/routes instead.
BASE  = 'gs://YOUR_BUCKET/porto/outputs/routes'   # <-- edit, or './outputs/routes'
SCALE = 'full'                                    # 'full' | 'mid' | 'sample'

def load(name):
    try:
        return pd.read_csv(f'{BASE}/{name}_{SCALE}.csv')
    except Exception as e:
        print('skip', name, '->', type(e).__name__)
        return pd.DataFrame()

M = {'A': load('clustering_top100'),
     'B': load('maximal_frequent_top100'),
     'C': load('graph_heavy_paths_top100'),
     'D': load('suffix_array_top100')}
Z  = load('activity_zones')
AN = load('anomalies_top50')
print({k: len(v) for k, v in M.items()}, '| zones:', len(Z), '| anomalies:', len(AN))

In [ ]:
# 3. Coverage: how many routes each method found at each length configuration.
#    An empty cell at >=20/40 km is a real finding, not a bug: it means no
#    corridor of that length is driven by enough distinct trips at this scale.
LENGTHS = [1, 3, 5, 10, 20, 40]
cov = pd.DataFrame(
    {m: {L: int((df['min_len_km'] == L).sum()) if len(df) else 0 for L in LENGTHS}
     for m, df in M.items()})
cov.index.name = 'min_len_km'
cov

In [ ]:
# 4. The map: every method x every length configuration as its own layer.
#    Use the layer control (top right) to switch between the six configurations.
DELIM = '>'
COLOURS = {'A': '#2e8b3d', 'B': '#1f5fbf', 'C': '#d1341c', 'D': '#7b2fbf'}
ROUTE_COL = {'A': 'subroute', 'B': 'subroute', 'C': 'route', 'D': 'subroute'}
LABEL = {'A': 'A clustering', 'B': 'B maximal-frequent',
         'C': 'C transition-graph', 'D': 'D suffix-array'}
SHOW_L = 3          # the band where every method has results

def polyline(cells):
    return [list(h3.h3_to_geo(c)) for c in str(cells).split(DELIM) if c]

m = folium.Map(location=(41.157, -8.629), zoom_start=12, tiles='cartodbpositron')

for key, df in M.items():
    if not len(df):
        continue
    for L in LENGTHS:
        sub = df[df.min_len_km == L].head(25)
        if not len(sub):
            continue
        fg = folium.FeatureGroup(name=f'{LABEL[key]} (>={L} km)', show=(L == SHOW_L))
        for _, r in sub.iterrows():
            pts = polyline(r[ROUTE_COL[key]])
            if len(pts) >= 2:
                folium.PolyLine(
                    pts, color=COLOURS[key], weight=3, opacity=0.7,
                    tooltip=f"{LABEL[key]} >={L}km | support={r['support']} "
                            f"| {r['length_km']:.2f} km").add_to(fg)
        fg.add_to(m)

if len(Z):
    zfg = folium.FeatureGroup(name='Activity zones (PageRank)', show=True)
    for _, z in Z.iterrows():
        folium.CircleMarker((z.lat, z.lon), radius=max(3, 14 - int(z['rank']) // 4),
                            color='#e8850c', fill=True, fill_opacity=0.5,
                            tooltip=f"zone #{z['rank']} pr={z['pagerank']:.5f}").add_to(zfg)
    zfg.add_to(m)

if len(AN):
    afg = folium.FeatureGroup(name='Anomalous routes', show=False)
    for _, a in AN.iterrows():
        folium.CircleMarker((a.start_lat, a.start_lon), radius=5, color='#555555',
                            fill=True, fill_opacity=0.7,
                            tooltip=f"score={a.anomaly_score} dist={a.dist_km}km").add_to(afg)
    afg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

In [ ]:
# 5. The deliverable table: top-100 popular long sub-routes for one configuration.
#    Change L to any of 1, 3, 5, 10, 20, 40.
L, METHOD = 3, 'B'
df = M[METHOD]
cols = [c for c in ['rank', 'support', 'length_km', 'n_cells', 'x_pct'] if c in df.columns]
df[df.min_len_km == L][cols].head(100)

In [ ]:
# 6. Do independent methods agree? Overlap of the cell sets they report at >=3 km.
#    Two methods with different failure modes converging on the same corridors is
#    the strongest evidence available that those corridors are real.
def cellsets(key, L=3):
    df = M[key]
    if not len(df):
        return []
    return [frozenset(str(r[ROUTE_COL[key]]).split(DELIM))
            for _, r in df[df.min_len_km == L].iterrows()]

def match_fraction(xs, ys, thr=0.5):
    if not xs or not ys:
        return float('nan')
    hit = sum(any(len(x & y) / len(x | y) >= thr for y in ys) for x in xs)
    return hit / len(xs)

present = [k for k in M if len(M[k])]
S = {k: cellsets(k) for k in present}
pd.DataFrame({c: {r: (1.0 if r == c else match_fraction(S[r], S[c])) for r in present}
              for c in present}).round(2)